### r estimation using

In [ ]:
using NPZ
using PyPlot, Statistics, Distributions
using BenchmarkTools
include("function/save_TNT_TN_w_bl.jl")   # ← ここに calc_TᵀN⁻¹T_terms / fwhm_tag 等がある前提
using PyCall
using LinearAlgebra
using SparseArrays
@pyimport healpy as hp
@pyimport numpy as np
using Printf

nside = 4
lmin  = 2
lmax  = 2nside            
spin  = 2               
sphenical_harmonics = set_sphenical_harmonics_terms!()

cov_mat_scal = zeros(2 * npix, 2 * npix)
cov_mat_tens = zeros(2 * npix, 2 * npix);

freq_bands = [40, 50, 60, 68, 78, 89, 100, 119, 140, 166, 195, 235, 280, 337, 402]
which_model = "d1 and s1"
r_input = 0.01
seed    = 101
num_I   = 2             
lmax_alm = lmax

# =========================
# WX の FWHM 設定（重要）
# =========================
# 倍率で指定（0.0 で bl なし） or fwhm_arcmin を直接指定
fwhm_pix_multiple = 0.0 #2.5          # ← ここを 0.0 にすると bl 無し
# fwhm_arcmin_override = 1100.0  # 明示したい場合はコメント解除

# [rad] を決定
fwhm_rad_wx = if @isdefined fwhm_arcmin_override
    fwhm_arcmin_override * (pi/10800)
else
    hp.nside2resol(nside) * fwhm_pix_multiple
end

# ファイル名タグ（save_TNT_TN.jl 内の fwhm_tag を使用）
fwhm_tag_str = fwhm_tag(fwhm_rad_wx; decimals=1)   # 例: "fwhm_1100p0arcmin"

# =========================
# WX をロード（新: タグ付き → 旧: タグ無しへフォールバック）
# =========================
function load_WX_with_tag(spin::Int, nside::Int, lmax::Int;
                          dir::AbstractString="../../WX_matrix",
                          fwhm_tag::AbstractString)
    # 新ファイル名（タグ付き）
    fname_tag = "WXmat_spin_$(spin)_nside_$(nside)_lmax_$(lmax)_$(fwhm_tag).npz"
    path_tag  = joinpath(dir, fname_tag)
    if isfile(path_tag)
        return npzread(path_tag), path_tag
    end
    # 旧ファイル名（タグ無し）にフォールバック
    fname_old = "WXmat_spin_$(spin)_nside_$(nside)_lmax_$(lmax).npz"
    path_old  = joinpath(dir, fname_old)
    if isfile(path_old)
        @warn "Tagged WX file not found. Falling back to: $path_old"
        return npzread(path_old), path_old
    end
    error("WX matrix file not found:\n  $path_tag\n  $path_old")
end

WX_data, WX_path = load_WX_with_tag(spin, nside, lmax; fwhm_tag=fwhm_tag_str)
Wmat = WX_data["Wmat"];  Xmat = WX_data["Xmat"]

# =========================
# マスク・パラメータセット
# =========================
mask_path = "../mask/P06_nside_$nside.fits"
mask = hp.read_map(mask_path)

N⁻¹_set     = Matrix{Float64}[]         # will be filled by set_truncate_N⁻¹!
TᵀN⁻¹_set   = Matrix{ComplexF64}[]
TᵀN⁻¹T_set  = Matrix{ComplexF64}[]
m_set       = Vector{Float64}[]
r_est       = 0.5

set_params = SetParams(freq_bands, which_model, r_input, seed, nside, num_I,
                       cov_mat_scal, cov_mat_tens, mask, m_set,
                       N⁻¹_set, TᵀN⁻¹_set, TᵀN⁻¹T_set, spin, lmax_alm)

fit_params = FitParams(-3, 1.5, 20.1, r_est)

# =========================
# T0 を作る（W/X は bl を含んだものを読み込んでいる想定）
# =========================
set_T0_matrix(set_params, sphenical_harmonics, Wmat, Xmat, mask_path)

# =========================
# N⁻¹ を用意 → TᵀN⁻¹ / TᵀN⁻¹T を bl 情報付きファイル名で保存
# =========================
set_truncate_N⁻¹!(set_params, lmin, lmax)

# save TᵀN⁻¹T
# WX（設計行列）側の bl は fwhm_rad_wx。Eビームを使っている旨を extra_tag で明記（任意）
calc_TᵀN⁻¹T_terms(set_params, sphenical_harmonics, mask_path, nside, lmin, lmax;
                   fwhm_rad_wx=fwhm_rad_wx, fname_decimals=1,
                   outdir_TNinv="T_N_inv",
                   outdir_TNinvT="T_N_inv_T",
                   extra_tag="")

println("Loaded WX from: $WX_path")
println("Saved TᵀN⁻¹ / TᵀN⁻¹T with tag: $(fwhm_tag_str)_WXbl")

Loaded WX from: ../../WX_matrix/WXmat_spin_2_nside_4_lmax_8_fwhm_0p0arcmin.npz
Saved TᵀN⁻¹ / TᵀN⁻¹T with tag: fwhm_0p0arcmin_WXbl
